# Pazos AI Turntable · v0.1.0
Illustrator → Drive → worker Colab → Drive → Illustrator. GitHub distribuye código y valida la suite; no recibe obras.

**Adaptador de prueba:** copias PNG rotuladas, sin generación multivista real. Usa CPU; selecciona GPU solo al instalar un modelo que la necesite.

Conecta la sesión y ejecuta todas las celdas. Autoriza Drive manualmente. Mantén abierta esta sesión interactiva. Un solo worker por carpeta Drive.
Colab Free no es un servicio permanente: puede desconectarse, restringe workers distribuidos y uso que evita su interfaz; este notebook no elude esos límites ni garantiza automatización desatendida. [FAQ oficial](https://research.google.com/colaboratory/faq.html).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/Pazos_AI_Turntable')
REPO = Path('/content/pazos-illustrator-tools')
BRANCH = 'main'
URL = 'https://github.com/ddepazos/ddepazos-illustrator-mcp-local.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ai-worker/requirements.txt')], check=True)
print('Código:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Drive:', DRIVE_ROOT)


## Iniciar worker
La celda permanece ejecutándose y procesa los trabajos que envías desde Illustrator. Detén la celda al terminar. Si la sesión se corta, vuelve a conectar y ejecutar; los trabajos interrumpidos se marcan como error y debes enviarlos de nuevo. No abras dos sesiones sobre la misma carpeta.
El panel comprueba el estado cada 5 segundos; Drive puede tardar más en sincronizar. No se muestran ni suben imágenes al notebook o GitHub automáticamente.


In [ ]:
process = subprocess.Popen([sys.executable, str(REPO / 'ai-worker/worker.py'), '--root', str(DRIVE_ROOT)])
try:
    result = process.wait()
    if result:
        raise RuntimeError(f'Worker terminó con código {result}; revisa los mensajes anteriores.')
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.terminate()
    print('Worker detenido.')
